In [17]:
# Use kernal base (Python 3.13.5)
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
import json
import os

print(f"Loading .env from: {find_dotenv()}")
load_dotenv()  # Load environment variables from .env file

Loading .env from: /Users/prernaroy/Library/CloudStorage/OneDrive-UNSW/Year 2/disso/dissertation/.env


True

In [5]:
# === Configuration ===
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # Load the API key from the .env file
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in your environment before running this notebook.")

client = OpenAI(api_key=OPENAI_API_KEY)
client = OpenAI()

# Testing OpenAI API

In [ ]:
# # Test a basic API request
# response = client.responses.create(
#     model="gpt-5.4-mini-2026-03-17", #dated model for reproducibility purposes
#     input="Write a one-sentence bedtimme story about a dragon."
# )

# print(response.output_text)

Under the silver moon, a gentle dragon curled around his tiny forest friends, warming them with his soft golden breath until they all drifted happily to sleep.


In [ ]:
# # Use a file URL as an input - example from OpenAI documentation
# response = client.responses.create(
#     model="gpt-5.4-mini-2026-03-17",
#     input=[
#         {
#             "role": "user",
#             "content": [
#                 {
#                     "type": "input_text",
#                     "text": "Analyze the letter and provide a summary of the key points.",
#                 },
#                 {
#                     "type": "input_file",
#                     "file_url": "https://www.berkshirehathaway.com/letters/2024ltr.pdf",
#                 },
#             ],
#         },
#     ]
# )

# print(response.output_text)

In [ ]:
# # Example of structured data extraction using Pydantic models
# class ResearchPaperExtraction(BaseModel):
#     title: str
#     authors: list[str]
#     abstract: str
#     keywords: list[str]

# response = client.responses.parse(
#     model="gpt-5.4-mini-2026-03-17",
#     input=[
#         {
#             "role": "system",
#             "content": "You are an expert at structured data extraction. You will be given unstructured text from a research paper and should convert it into the given structure.",
#         },
#         {
#             "role": "user", "content": "..."},
#     ],
#     text_format=ResearchPaperExtraction,
# )

# research_paper = response.output_parsed

In [ ]:
# # Define the structured schema we want the model to return.
# class ResearchPaperExtraction(BaseModel):
#     title: str
#     authors: list[str]
#     abstract: str
#     keywords: list[str]

# client = OpenAI()

# # Define the parsing 
# response = client.responses.parse(
#     model="gpt-5.4-mini-2026-03-17",
#     input=[
#         {
#             "role": "system",
#             "content": "You are an expert at structured data extraction. You will be given unstructured text from a research paper and should convert it into the given structure.",
#         },
#         {
#             "role": "user",
#             "content": [
#                 {
#                     "type": "input_text",
#                     "text": "Please extract title, authors, abstract, and keywords from the linked paper."
#                 },
#                 {
#                     "type": "input_file",
#                     "file_url": "https://arxiv.org/pdf/2303.08774.pdf",
#                 }
#             ]
#         }

#     ],
#     text_format=ResearchPaperExtraction,
# )

# research_paper = response.output_parsed
# print(research_paper)

title='GPT-4 Technical Report' authors=['OpenAI'] abstract='We report the development of GPT-4, a large-scale, multimodal model which can accept image and text inputs and produce text outputs. While less capable than humans in many real-world scenarios, GPT-4 exhibits human-level performance on various professional and academic benchmarks, including passing a simulated bar exam with a score around the top 10% of test takers. GPT-4 is a Transformer-based model pre-trained to predict the next token in a document. The post-training alignment process results in improved performance on measures of factuality and adherence to desired behavior. A core component of this project was developing infrastructure and optimization methods that behave predictably across a wide range of scales. This allowed us to accurately predict some aspects of GPT-4’s performance based on models trained with no more than 1/1,000th the compute of GPT-4.' keywords=['GPT-4', 'large-scale multimodal model', 'Transforme

# Coronial Reports Testing

Following the success of this method, I will now use this to test with a Coroner's report with the following variables:
1. Mental health condition flag 
2. Mental health label 
3. Developmental disability flag
4. Developmental disability label

In [ ]:
# Create Pydantic model for structured data extraction of Coronial findings and recommendations.
class VariablesToExtract(BaseModel):
    file_number: str
    mental_health_flag: bool
    mental_health_labels: list[str]
    developmental_disability_flag: bool
    developmental_disability_labels: list[str]

In [13]:
system_content = "You are an expert at structured data extraction. You will be given unstructured text from a coronial findings and recommendations report and should convert it into the given structure."

In [14]:
user_text = "Extract the following information from the coronial finding and recommendations report: file number, mental health flag, mental health labels, developmental disability flag, and developmental disability labels. Flags should be set to True if any diagnosed mental health conditions are present. If a flag is true, the corresponding labels should contain the specific mental health conditions and/or developmental disability diagnosed. If no conditions are present, the flag should be set to False and the labels should be an empty list."

In [15]:
file_url = "https://coroners.nsw.gov.au/documents/findings/2019/CD%20Findings.pdf"

In [18]:
client = OpenAI()

# Define the parsing request to extract structured data from a coronial finding and recommendations report.
response = client.responses.parse(
    model="gpt-5.4-mini-2026-03-17",
    input=[
        {
            "role": "system",
            "content": system_content,
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": user_text
                },
                {
                    "type": "input_file",
                    "file_url": file_url,
                }
            ]
        }

    ],
    text_format=VariablesToExtract,
)

report_variables_json = response.output_parsed.model_dump_json(indent=2)
print(report_variables_json)

{
  "file_number": "2017/286401",
  "mental_health_flag": true,
  "mental_health_labels": [
    "schizophrenia",
    "schizoaffective disorder",
    "suicidality",
    "mental health issues",
    "possible hypomania"
  ],
  "developmental_disability_flag": false,
  "developmental_disability_labels": []
}


Managed to get the output into JSON reliably for 4 variables. This will now be extended to more variables, including some text summary variables to see what the output is like. 

In [ ]:
# Create Pydantic model for structured data extraction of Coronial findings and recommendations.
class VariablesToExtractExtended(BaseModel):
    file_number: str
    mental_disorder_flag: bool
    mental_disorder_labels: list[str]
    cognitive_disability_flag: bool
    cognitive_disability_labels: list[str]
    date_of_findings: str
    date_of_death: str
    age_at_death: int # Potential change to str for age estimates
    sex: str
    country_of_birth: str
    place_of_residence: str
    profession: str
    level_of_education: str
    comorbidities: list[str]
    medications: list[str]
    known_health_issues: list[str]
    place_of_death: str
    definitive_cause_of_death: str
    manner_of_death: str
    site_of_incident: str
    potential_causes_of_death: list[str]
    coroners_recommendations_summary: str
    custody_served: bool
    use_of_interpreters: bool
    preventable_death_flag: bool
    non_english_speaking_background_flag: bool
    circumstances_of_death_summary: str
    toxicology_results_summary: str
    domestic_violence_flag: bool

In [ ]:
system_content = "You are an expert at structured data extraction. You will be given unstructured text from a coronial findings and recommendations report and must convert it into the given structure."

# Include a sentence here to specify what is a MHC and DD

In [ ]:
user_text = """Extract the following medical and demographic information from the given coronial findings and recommmendations report. Only extract information present in the report. If a field is not present or unknown, return an empty string, an empty list for list fields, and False for boolean flags. Flags must be True if verifiably present in the report with supporting evidence such as doctor diagnoses, or stated clearly in the inquest report from a verifiable source. Outputs must be accurate and concise; do not provide additional commentary or explanation. Labels should be short, specific, and present in the report; do not paraphrase directly from the report. For summary variables, provide a summary of the key points if present, otherwise return an empty string. Do not provide potential speculative information when definitive information is in the report. Do not provide additional information beyond what is requested. Do not repeat information that is already in another variable. Use ICD-10 to accurately identify mental disorders and cognitive disabilities.

Extract the following fields: file_number, mental_disorder_flag, mental_disorder_labels, cognitive_disability_flag, cognitive_disability_labels, date_of_findings, date_of_death, age_at_death, sex, country_of_birth, place_of_residence, profession, level_of_education, comorbidities, medications, known_health_issues, place_of_death, definitive_cause_of_death, manner_of_death, site_of_incident, potential_causes_of_death, coroners_recommendations_summary, custody_served, use_of_interpreters, preventable_death_flag, non_english_speaking_background_flag, circumstances_of_death_summary, toxicology_results_summary, and domestic_violence_flag."""

# Research whether providing just sentence or the whole report (should be whole report) is sufficient for examples in few-shot 
# Few examples where I only give the sentence 
# Systematic approach on how to test this -- research 
# Reference Ida's dissertation - used whole discharge summary instead of just specific sections
# 

In [40]:
file_url = "https://coroners.nsw.gov.au/documents/findings/2022/Inquest_into_the_death_of_Riley_Christopher_Shortland.pdf"

In [ ]:
client = OpenAI()

# Define the parsing request to extract structured data from a coronial finding and recommendations report.
response = client.responses.parse(
    model="gpt-5.4-mini-2026-03-17",
    input=[
        {
            "role": "system",
            "content": system_content,
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": user_text
                },
                {
                    "type": "input_file",
                    "file_url": file_url,
                }
            ]
        }

    ],
    text_format=VariablesToExtractExtended,
)

report_variables_json = response.output_parsed.model_dump_json(indent=2)
print(report_variables_json)

# Few-shot examples to specify what is MHC and disability 
# Check out Aaron's dissertation - he specifies in the prompt; use as an example of "do this and not that"

{
  "file_number": "2017/00335331",
  "mental_disorder_flag": true,
  "mental_disorder_labels": [
    "ASD",
    "ADHD"
  ],
  "cognitive_disability_flag": true,
  "cognitive_disability_labels": [
    "global developmental delay"
  ],
  "date_of_findings": "21 October 2022",
  "date_of_death": "5 November 2017",
  "age_at_death": 8,
  "sex": "Male",
  "country_of_birth": "",
  "place_of_residence": "Woy Woy, NSW",
  "profession": "",
  "level_of_education": "",
  "comorbidities": [
    "Autism Spectrum Disorder",
    "Global Development Delay",
    "Attention Deficit Hyperactivity Disorder"
  ],
  "medications": [
    "amphetamine",
    "risperidone",
    "citalopram"
  ],
  "known_health_issues": [
    "predominantly non-verbal",
    "limited speech and communication skills",
    "required close supervision at all times"
  ],
  "place_of_death": "M1 Pacific Motorway near Cameron Park NSW",
  "definitive_cause_of_death": "multiple injuries sustained when he was hit by a truck",
  "mann

Using this example, I see that there is an issue that needs to be addressed: the accurate definition of what is considered a mental disorder versus a developmental or cognitive disability. In different runs of this input, the output will sometimes only mark `cognitive_disability_flag` as `True` and `mental_disorder_flag` as `False`. However, in other runs, it considers ASD and ADHD to be mental disorders, while the only cognitive disability is "global developmental delay". 

This is something to think about, but for now I will focus on building the evaluation pipeline using Promptfoo, which is now OpenAI's preferred way to conduct evaluations: https://developers.openai.com/cookbook/examples/evaluation/moving-from-openai-evals-to-promptfoo

## Evaluation Pipeline Setup

The goal of the evaluation pipeline is to:
1. Version prompts and record the model output for each prompt used (store the prompt and the model output for future reference).
2. Determine the formatting accuracy of the model output (it should be JSON). 
3. Measure the accuracy and relevancy of the model output (the value produced for each variable extracted). 